# Lab | Neural Networks Fundamentals

## Task 1 — A Single Neuron in NumPy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)

data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

w = np.random.randn(30) * 0.01
b = np.random.randn() * 0.01


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def forward(x, w, b):
    return sigmoid(x @ w + b)


single_neuron_preds = forward(X_test[:5], w, b)

print('Weights shape:', w.shape)
print('Bias:', b)
print('Predicted probabilities for the first 5 test rows:')
print(single_neuron_preds)

In machine-learning terms, this single neuron is a logistic regression model for binary classification.

## Task 2 — A Two-Layer MLP in NumPy

In [ ]:
class NumpyMLP:
    def __init__(self, input_size=30, hidden_size=8, output_size=1):
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2 / hidden_size)
        self.b2 = np.zeros((1, output_size))

    def relu(self, z):
        return np.maximum(0, z)

    def forward(self, X):
        z1 = X @ self.W1 + self.b1
        a1 = self.relu(z1)
        z2 = a1 @ self.W2 + self.b2
        return sigmoid(z2)


mlp = NumpyMLP(input_size=30, hidden_size=8, output_size=1)
numpy_preds = mlp.forward(X_test)

print('Output shape:', numpy_preds.shape)
print('First 5 predictions:')
print(numpy_preds[:5])

The output shape is `(N, 1)` because the network has one sigmoid output neuron per input row, which is exactly one predicted probability for the positive class in binary classification.

## Task 3 — The Same Network in PyTorch

In [ ]:
class TorchMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(30, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, X):
        X = self.relu(self.fc1(X))
        X = self.sigmoid(self.fc2(X))
        return X


torch_mlp = TorchMLP()

torch_mlp.fc1.weight.data = torch.from_numpy(mlp.W1.T).float()
torch_mlp.fc1.bias.data = torch.from_numpy(mlp.b1.ravel()).float()
torch_mlp.fc2.weight.data = torch.from_numpy(mlp.W2.T).float()
torch_mlp.fc2.bias.data = torch.from_numpy(mlp.b2.ravel()).float()

X_test_tensor = torch.from_numpy(X_test).float()

with torch.no_grad():
    torch_preds = torch_mlp(X_test_tensor).numpy()

max_abs_diff = np.max(np.abs(numpy_preds - torch_preds))

print('First 5 NumPy predictions:')
print(numpy_preds[:5])
print('First 5 PyTorch predictions:')
print(torch_preds[:5])
print('Maximum absolute difference:', max_abs_diff)

The NumPy and PyTorch networks produce the same predictions because they use the same architecture, the same copied weights and biases, and the same input data.

## Task 4 — Activation Function Experiment

In [ ]:
class ActivationMLP(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.fc1 = nn.Linear(30, 8)
        self.activation = activation
        self.fc2 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, X):
        z1 = self.fc1(X)
        a1 = self.activation(z1)
        z2 = self.fc2(a1)
        output = self.sigmoid(z2)
        return output, z1, a1


activations = {
    'Sigmoid': nn.Sigmoid(),
    'Tanh': nn.Tanh(),
    'ReLU': nn.ReLU(),
}

activation_results = {}

for name, activation in activations.items():
    model = ActivationMLP(activation)
    with torch.no_grad():
        preds, z1, a1 = model(X_test_tensor)
    activation_results[name] = {
        'preds': preds.numpy(),
        'z1': z1.numpy(),
        'a1': a1.numpy(),
    }

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, values) in zip(axes, activation_results.items()):
    ax.hist(values['z1'].ravel(), bins=30)
    ax.set_title(f'{name}: pre-activations')
    ax.set_xlabel('z1')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, values) in zip(axes, activation_results.items()):
    ax.hist(values['a1'].ravel(), bins=30)
    ax.set_title(f'{name}: post-activations')
    ax.set_xlabel('a1')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

sigmoid_a1 = activation_results['Sigmoid']['a1']
tanh_a1 = activation_results['Tanh']['a1']
relu_a1 = activation_results['ReLU']['a1']

sigmoid_saturated = np.mean((sigmoid_a1 < 0.05) | (sigmoid_a1 > 0.95))
tanh_saturated = np.mean(np.abs(tanh_a1) > 0.95)
relu_inactive = np.mean(relu_a1 == 0)

print('Sigmoid saturated fraction:', sigmoid_saturated)
print('Tanh saturated fraction:', tanh_saturated)
print('ReLU inactive fraction:', relu_inactive)

Sigmoid and tanh can become saturated when their outputs are close to their flat regions, which can make gradients very small. ReLU is often a better default for hidden layers because it keeps positive activations in a non-saturated linear region, making optimization easier, although some ReLU units may be inactive when their output is exactly zero.